In [1]:
!pip uninstall -y -q \
transformers peft trl accelerate bitsandbytes huggingface_hub

!pip install -q -U \
transformers==4.51.3 \
tokenizers==0.21.1 \
huggingface_hub==0.36.2 \
accelerate==1.10.1 \
peft==0.19.1 \
trl==0.18.1 \
datasets==4.0.0 \
bitsandbytes==0.47.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 73.4 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 78.1 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.9/374.9 kB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 29.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.3/366.3 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 494.8/494.8 kB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 MB 30.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 10.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts

In [2]:
import torch
import transformers
import peft
import trl
import bitsandbytes
import accelerate

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("peft:", peft.__version__)
print("trl:", trl.__version__)
print("bnb:", bitsandbytes.__version__)
print("accelerate:", accelerate.__version__)

print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

2026-05-10 11:05:48.041680: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778411148.432385      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778411148.548230      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778411149.518116      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778411149.518149      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778411149.518152      57 computation_placer.cc:177] computation placer alr

torch: 2.10.0+cu128
transformers: 4.51.3
peft: 0.19.1
trl: 0.18.1
bnb: 0.47.0
accelerate: 1.10.1
CUDA: True
GPU: Tesla T4


In [3]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"


  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-f5gq6u3e/unsloth_33dad2abdd5d4ae5893ae1cf866f459c
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-f5gq6u3e/unsloth_33dad2abdd5d4ae5893ae1cf866f459c
  Resolved https://github.com/unslothai/unsloth.git to commit b3640802253f64117ee228718be7fab32e47aa5f
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
INFO: pip is looking at multiple versions of transformers to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 12.3 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 428.0/428.0 kB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 73.4 MB/s eta 0:00:00:00:01

In [4]:
#!/usr/bin/env python3
"""
Fine-tune Qwen2.5-7B-Instruct — English→Vietnamese (LoRA / QLoRA / Unsloth). Một file duy nhất.

So với script LoRA 1-GPU trên GitHub Qwen (Qwen-7B + finetune.py):
- Đã lấy: bf16, grad_accum=8, cosine, adam_beta2=0.95, lr≈3e-4, weight_decay=0.1, warmup_ratio=0.01,
  CUDA_DEVICE_MAX_CONNECTIONS=1 (ổn định kernel/driver khi train transformer trên GPU).
- Giữ khác hợp lý: max_seq_len mặc định 2048 (recipe Qwen dùng 512 cho hội thoại ngắn; prompt dịch của bạn ~1k+ ký tự → 512 sẽ cắt mất glossary/context).
- Giữ eval định kỳ (recipe Qwen có thể tắt eval cho nhanh); bạn có valid_stratified → nên eval.
- Không dùng: lazy_preprocess (API riêng finetune.py), DeepSpeed+fp16 (bạn dùng bf16/QLoRA trên T4).
- Có thể thử per_device_batch_size=2 như Qwen nếu đủ VRAM (đang mặc định 1 cho QLoRA an toàn).

Trên Kaggle:

  pip install -r requirements-train.txt
  # hoặc: pip install -U "huggingface_hub>=0.26" transformers ...  (Kaggle/Colab hay có hub cũ)

  accelerate launch --num_processes 2 --multi_gpu --mixed_precision bf16 train_qwen25_en_vi.py
  python train_qwen25_en_vi.py   # 1 GPU, HF + PEFT + QLoRA

  # 1 GPU, Unsloth (nhanh hơn, cần: pip install unsloth) — không dùng chung multi-GPU accelerate
  python train_qwen25_en_vi.py --use_unsloth

Biến môi trường: MODEL_ID, TRAIN_JSONL, VALID_JSONL, OUTPUT_DIR hoặc cờ --model_id / --train_path / ...

JSONL: mỗi dòng {"prompt": "...", "completion": "..."}.

Lỗi thường gặp:
- ``cannot import name 'is_offline_mode' from 'huggingface_hub'``: nâng hub — ``pip install -U "huggingface_hub>=0.26"`` rồi restart kernel (không cần pytorch-quantization).
- ``CUDA out of memory`` tại ``prepare_model_for_kbit_training`` với **2×GPU** nhưng **một process** (không accelerate): toàn bộ model đang nằm trên GPU0 —
  script tự ``max_memory`` chia layer sang cả hai GPU; hoặc chạy ``accelerate launch`` (DDP), hoặc ``export PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True``.
- ``SFTConfig ... unexpected keyword 'max_seq_length'``: môi trường TRL cũ — script tự chọn ``max_length`` hoặc ``max_seq_length`` theo phiên bản.
- ``SFTTrainer ... unexpected keyword 'packing'``: script **không** truyền ``packing`` (một số môi trường TRL lỗi dù inspect thấy tham số).
"""

from __future__ import annotations

import argparse
import gc
import inspect
import json
import os
from pathlib import Path

import torch
from datasets import Dataset
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer


def _sft_max_len_kw(max_len: int) -> dict:
    """Tương thích TRL: bản cũ dùng max_length, bản mới dùng max_seq_length."""
    params = inspect.signature(SFTConfig.__init__).parameters
    if "max_seq_length" in params:
        return {"max_seq_length": max_len}
    if "max_length" in params:
        return {"max_length": max_len}
    return {}


def _sft_trainer_extra_kw(tokenizer, formatting_func) -> dict:
    """Tương thích SFTTrainer: processing_class vs tokenizer; formatting_func.

    Không truyền ``packing``: nhiều bản TRL/Kaggle lệch (signature có nhưng __init__ báo lỗi).
    Mặc định không pack khi dùng formatting_func.
    """
    p = inspect.signature(SFTTrainer.__init__).parameters
    kw: dict = {}
    if "processing_class" in p:
        kw["processing_class"] = tokenizer
    elif "tokenizer" in p:
        kw["tokenizer"] = tokenizer
    if "formatting_func" in p:
        kw["formatting_func"] = formatting_func
    return kw


KAGGLE_MODEL_DIR = "/kaggle/input/models/qwen-lm/qwen2.5/transformers/7b-instruct/1"
KAGGLE_DATA_DIR = "/kaggle/input/datasets/thachng/dataset-qwen"
KAGGLE_OUTPUT_DIR = "/kaggle/working/qwen25-7b-en-vi-lora"

LORA_TARGET_MODULES = [
    "q_proj",
    "k_proj",
    "v_proj",
    "o_proj",
    "gate_proj",
    "up_proj",
    "down_proj",
]


def _on_kaggle() -> bool:
    return Path("/kaggle/working").is_dir()


def default_paths() -> dict:
    if _on_kaggle():
        return {
            "model_id": os.environ.get("MODEL_ID", KAGGLE_MODEL_DIR),
            "train_path": Path(
                os.environ.get("TRAIN_JSONL", f"{KAGGLE_DATA_DIR}/train_stratified.json")
            ),
            "valid_path": Path(
                os.environ.get("VALID_JSONL", f"{KAGGLE_DATA_DIR}/valid_stratified.json")
            ),
            "output_dir": Path(os.environ.get("OUTPUT_DIR", KAGGLE_OUTPUT_DIR)),
        }
    return {
        "model_id": os.environ.get("MODEL_ID", "Qwen/Qwen2.5-7B-Instruct"),
        "train_path": Path(os.environ.get("TRAIN_JSONL", "./train_stratified.json")),
        "valid_path": Path(os.environ.get("VALID_JSONL", "./valid_stratified.json")),
        "output_dir": Path(os.environ.get("OUTPUT_DIR", "./qwen25-7b-en-vi-lora")),
    }


def load_jsonl(path: Path) -> list[dict]:
    rows: list[dict] = []
    with path.open(encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rows.append(json.loads(line))
    return rows


def build_hf_dataset(rows: list[dict]) -> Dataset:
    return Dataset.from_dict(
        {
            "prompt": [r["prompt"] for r in rows],
            "completion": [r["completion"] for r in rows],
        }
    )


def is_distributed() -> bool:
    return int(os.environ.get("WORLD_SIZE", "1")) > 1


def main() -> None:
    d = default_paths()
    parser = argparse.ArgumentParser()
    parser.add_argument("--train_path", type=Path, default=d["train_path"])
    parser.add_argument("--valid_path", type=Path, default=d["valid_path"])
    parser.add_argument("--output_dir", type=Path, default=d["output_dir"])
    parser.add_argument("--model_id", type=str, default=d["model_id"])
    parser.add_argument("--max_seq_len", type=int, default= 512)
    parser.add_argument("--num_epochs", type=int, default= 2)
    parser.add_argument("--batch_size", type=int, default=1)
    parser.add_argument("--grad_accum", type=int, default=8)
    parser.add_argument("--lr", type=float, default=3e-4)
    parser.add_argument("--lora_r", type=int, default=16)
    parser.add_argument("--lora_alpha", type=int, default=32)
    parser.add_argument("--lora_dropout", type=float, default=0.05)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument(
        "--no_qlora",
        action="store_true",
        help="Tắt 4-bit (VRAM đủ hoặc CPU). Không dùng chung --use_unsloth.",
    )
    parser.add_argument(
        "--use_unsloth",
        action="store_true",
        help="Unsloth (1 GPU CUDA, pip install unsloth). Multi-GPU: bỏ cờ này, dùng accelerate + script không Unsloth.",
    )
    # Jupyter/Colab: %run chèn -f /path/kernel-....json → parse_known_args bỏ qua.
    args, _unknown = parser.parse_known_args()

    if args.use_unsloth and args.no_qlora:
        raise SystemExit("Không dùng đồng thời --use_unsloth và --no_qlora.")
    if args.use_unsloth:
        if not torch.cuda.is_available():
            raise SystemExit("--use_unsloth cần CUDA.")
        if is_distributed():
            raise SystemExit(
                "--use_unsloth chỉ hỗ trợ 1 process / 1 GPU. Multi-GPU: chạy không --use_unsloth + accelerate launch."
            )

    if torch.cuda.is_available():
        os.environ.setdefault("CUDA_DEVICE_MAX_CONNECTIONS", "1")
        os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

    dist = is_distributed()
    use_cuda = torch.cuda.is_available()
    use_qlora = use_cuda and not args.no_qlora and not args.use_unsloth
    use_unsloth = bool(args.use_unsloth)

    if dist:
        local_rank = int(os.environ["LOCAL_RANK"])
        torch.cuda.set_device(local_rank)
    else:
        local_rank = 0

    torch.manual_seed(args.seed)
    if use_cuda:
        torch.cuda.manual_seed_all(args.seed)

    if use_cuda:
        torch_dtype = torch.float16
        if local_rank == 0:
            print("CUDA:", torch.cuda.get_device_name(0), "| nGPU:", torch.cuda.device_count())
            print("Model:", args.model_id)
            print(
                "Backend:",
                "Unsloth" if use_unsloth else "HF+PEFT",
                "| QLoRA (4-bit):",
                use_qlora or use_unsloth,
                "| distributed:",
                dist,
            )
    else:
        torch_dtype = torch.bfloat16 if torch.cpu.is_bf16_supported() else torch.float32
        if use_unsloth:
            raise SystemExit("Unsloth cần GPU.")
        print("CPU mode | torch dtype:", torch_dtype, "(không dùng quant)")

    train_rows = load_jsonl(args.train_path)
    valid_rows = load_jsonl(args.valid_path)
    train_ds = build_hf_dataset(train_rows)
    valid_ds = build_hf_dataset(valid_rows)
    if local_rank == 0:
        print(f"Train samples: {len(train_ds)} | Valid samples: {len(valid_ds)}")

    model = None
    tokenizer = None

    if use_unsloth:
        from unsloth import FastLanguageModel

        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name=args.model_id,
            max_seq_length=args.max_seq_len,
            dtype=None,
            load_in_4bit=True,
        )
        model = FastLanguageModel.get_peft_model(
            model,
            r=args.lora_r,
            target_modules=LORA_TARGET_MODULES,
            lora_alpha=args.lora_alpha,
            lora_dropout=args.lora_dropout,
            bias="none",
            use_gradient_checkpointing="unsloth",
            random_state=args.seed,
        )
    else:
        tokenizer = AutoTokenizer.from_pretrained(args.model_id, trust_remote_code=True)
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
            model.config.pad_token_id = tokenizer.pad_token_id
            model.config.bos_token_id = tokenizer.bos_token_id
            model.config.eos_token_id = tokenizer.eos_token_id
        tokenizer.padding_side = "right"

        quant_config = None
        if use_qlora:
            quant_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_compute_dtype=torch_dtype,
                bnb_4bit_use_double_quant=True,
                bnb_4bit_quant_type="nf4",
            )

        if use_cuda:
            device_map = "auto"
        else:
            device_map = {"": "cpu"}

        load_kw: dict = {
            "device_map": device_map,
            "trust_remote_code": True,
            "low_cpu_mem_usage": True,
        }
        if quant_config is not None:
            load_kw["quantization_config"] = quant_config
        else:
            load_kw["torch_dtype"] = torch_dtype

        # 2×GPU nhưng 1 process: device_map="auto" mặc định có thể dồn hết lên cuda:0 → OOM khi prepare kbit (upcast fp32).
        if use_cuda and use_qlora and not dist and torch.cuda.device_count() > 1:
            n = torch.cuda.device_count()
            per_gib = 11
            max_memory = {i: f"{per_gib}GiB" for i in range(n)}
            max_memory["cpu"] = "64GiB"
            load_kw["max_memory"] = max_memory
            load_kw["device_map"] = "auto"
            if local_rank == 0:
                print(
                    f"QLoRA: chia model lên {n} GPU (max_memory ~{per_gib}GiB/GPU) để tránh OOM trên một T4."
                )

        model = AutoModelForCausalLM.from_pretrained(args.model_id, **load_kw)
        model.config.use_cache = False

        if use_qlora:
            gc.collect()
            torch.cuda.empty_cache()
            
            model.gradient_checkpointing_enable()
        else:
            model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})

        peft_config = LoraConfig(
            task_type=TaskType.CAUSAL_LM,
            r=args.lora_r,
            lora_alpha=args.lora_alpha,
            lora_dropout=args.lora_dropout,
            bias="none",
            target_modules=LORA_TARGET_MODULES,
        )
        model = get_peft_model(model, peft_config)

    assert model is not None and tokenizer is not None
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        
    tokenizer.padding_side = "right"

    if local_rank == 0:
        model.print_trainable_parameters()

    def to_text(example):
        messages = [
            {"role": "user", "content": example["prompt"]},
            {"role": "assistant", "content": example["completion"]},
        ]
    
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
        )
    
        return {"text": text}
    
    
    train_ds = train_ds.map(
        to_text,
        remove_columns=train_ds.column_names,
    )
    
    valid_ds = valid_ds.map(
        to_text,
        remove_columns=valid_ds.column_names,
    )

    if use_unsloth:
        optim = "adamw_8bit"
    elif use_qlora:
        optim = "paged_adamw_8bit"
    else:
        optim = "adamw_torch"

    training_args = SFTConfig(
        output_dir=str(args.output_dir),
        num_train_epochs=args.num_epochs,
        per_device_train_batch_size=args.batch_size,
        per_device_eval_batch_size=args.batch_size,
        gradient_accumulation_steps=args.grad_accum,
        learning_rate=args.lr,
        weight_decay=0.1,
        adam_beta2=0.95,
        warmup_ratio=0.01,
        lr_scheduler_type="cosine",
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=200,
        save_strategy="steps",
        save_steps=1000,
        save_total_limit=10,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        bf16=use_cuda and torch_dtype == torch.bfloat16,
        fp16=False,
        gradient_checkpointing=True,
        optim=optim,
        **_sft_max_len_kw(args.max_seq_len),
        report_to="none",
        seed=args.seed,
        dataloader_num_workers= 4,
        dataloader_pin_memory=bool(use_cuda),
        ddp_find_unused_parameters=False,
    )

    trainer = SFTTrainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=valid_ds,
        processing_class=tokenizer,
    )

    trainer.train()
    if local_rank == 0:
        trainer.save_model(str(args.output_dir))
        tokenizer.save_pretrained(str(args.output_dir))
        print("Saved adapter + tokenizer to", args.output_dir)


if __name__ == "__main__":
    main()


RuntimeError: Failed to import trl.trainer.sft_config because of the following error (look up to see its traceback):
Failed to import transformers.training_args because of the following error (look up to see its traceback):
cannot import name 'is_torch_greater_or_equal_than_2_8' from 'transformers.pytorch_utils' (/usr/local/lib/python3.12/dist-packages/transformers/pytorch_utils.py)